In [8]:
import pandas as pd

X_train = pd.read_csv('../Data/X_train.csv')

print(X_train.columns.tolist())

pd.DataFrame(X_train.columns, columns=['Fitur'])

['VendorID', 'passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID', 'DOLocationID', 'pickup_hour', 'day_of_week', 'is_weekend']


,Fitur
0,VendorID
1,passenger_count
2,trip_distance
3,RatecodeID
4,PULocationID
5,DOLocationID
6,pickup_hour
7,day_of_week
8,is_weekend


In [9]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor

from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [10]:
X_train = pd.read_csv('../Data/X_train.csv')
X_test = pd.read_csv('../Data/X_test.csv')
y_train = pd.read_csv('../Data/y_train.csv')
y_test = pd.read_csv('../Data/y_test.csv')

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(3157358, 9)
(789340, 9)
(3157358, 1)
(789340, 1)


In [11]:
#Change target to 1 dimension
y_train = y_train.squeeze()
y_test = y_test.squeeze()

print(y_train.shape)
print(y_test.shape)

(3157358,)
(789340,)


In [12]:
#Determine the parameters to be tested
param_dist = {

    'n_estimators':[100,200,300],
    'max_depth':[4,6,8],
    'learning_rate':[0.03,0.05,0.1],
    'subsample':[0.8,0.9,1.0],
    'colsample_bytree':[0.8,1.0]
}

In [17]:
# Hyperparameter tuning

xgb = XGBRegressor(

    objective='reg:squarederror',
    random_state=42,
    tree_method='hist'

)

random_search = RandomizedSearchCV(

    estimator=xgb,
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring='r2',
    n_jobs=1,
    verbose=1,
    random_state=42

)

random_search.fit(X_train,y_train)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...state=42, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.8, 1.0], 'learning_rate': [0.03, 0.05, ...], 'max_depth': [4, 6, ...], 'n_estimators': [100, 200, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturn

In [18]:
#Best parameters
print(random_search.best_params_)
print(random_search.best_score_)

{'subsample': 1.0, 'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
0.8308500935147785


In [19]:
#Save the best model
best_model = random_search.best_estimator_
print(best_model)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=1.0, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=8, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=200, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)


In [20]:
#Model evaluation
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test,y_pred)
rmse = np.sqrt(mean_squared_error(y_test,y_pred))
r2 = r2_score(y_test,y_pred)

print('MAE :',mae)
print('RMSE :',rmse)
print('R2 :',r2)

MAE : 3.428722244597203
RMSE : 5.498632681592765
R2 : 0.8317488558371929


In [ ]:
#important features
importance = pd.DataFrame({

    'Feature': X_train.columns,
    'Importance': best_model.feature_importances_

})

importance = importance.sort_values(
    by='Importance',
    ascending=False
).reset_index(drop=True)

importance

,Feature,Importance
0,trip_distance,0.494684
1,RatecodeID,0.333055
2,is_weekend,0.060289
3,pickup_hour,0.055889
4,day_of_week,0.021052
5,DOLocationID,0.014050
6,PULocationID,0.011008
7,VendorID,0.006838
8,passenger_count,0.003135


### Insight

1. Hyperparameter tuning menggunakan RandomizedSearchCV menghasilkan performa yang baik dengan R² sebesar 0,8317, MAE sebesar 3,43 menit, dan RMSE sebesar 5,50 menit.

2. Model mampu menjelaskan sekitar 83,17% variasi durasi perjalanan taksi dengan tingkat kesalahan prediksi yang relatif rendah.

3. XGBoost dipilih sebagai model final karena memberikan keseimbangan yang baik antara akurasi, efisiensi komputasi, dan kemampuan menangani dataset berukuran besar.

4. Meskipun Random Forest memiliki performa sedikit lebih tinggi, selisihnya relatif kecil, sementara XGBoost jauh lebih efisien dari sisi waktu komputasi sehingga lebih sesuai untuk implementasi operasional.

5. Analisis feature importance menunjukkan bahwa trip_distance, RatecodeID, dan faktor waktu (pickup_hour, day_of_week, dan is_weekend) merupakan faktor utama yang memengaruhi durasi perjalanan, sedangkan VendorID dan passenger_count memiliki pengaruh yang relatif kecil.